# [1교시]

### 프롬프트 템플릿 Agent tool rag에서 표준
- role: LLM이 어떤 역할을 할 지 정함
- instruction: 답변 규칙과 형식을 정리
- context: 데이터베이스 검색 결과처럼 답변에 참고 할 실제 정보
- query:실제 질문

In [1]:
# 사용자 질문
# 적절한 tool 호출 -> 라우터 ( Fast API )
# 수집한 정보를 -> LLM 전달
# 최종답변은 LLM

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv(override=True)

api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    raise EnvironmentError('openai api key .....')

class OpenAILLM:
    def __init__(self,model:str = 'gpt-5.4-nano'):
        self.client = OpenAI(api_key=api_key)
        self.model = model
    def generate(self, prompt:str) -> str:
        response = self.client.chat.completions.create(
            model = self.model,
            messages =[
                {"role":"system", "content":"You are an ecomerce recommendation assistant, Return only valid JSON"},
                {"role":"user", "content":prompt}
            ],
            temperature=0,
            response_format={'type':'json_object'}
        )
        return response.choices[0].message.content
llm = OpenAILLM()
llm.model  

'gpt-5.4-nano'

In [3]:
# 데이터
do_search_results = [
    {"id":"p1", "name":"고성능 노트북","category":"가전"},
    {"id":"p2", "name":"사무용 랩탑","category":"가전"},
    {"id":"p3", "name":"미러리스 카메라","category":"가전"},
]
context_string = ""
for item in do_search_results:
    context_string += f"- 상품명:{item['name']} | 카테고리:{item['category']}"

In [4]:
# 프롬프트 템플릿 - 고정된 구조
from  textwrap import dedent  #  들여쓰기 indent를 제거
user_query = '코딩할 때 쓸만한 노트북 하나 추천해줘, 믿을만한 제품으로'
raw_prompt = dedent(f'''
                    당신은 이커머스 전문 AI 추천 어시스턴트입니다.
                    아래 컨텍스트를 참고하여 사용자의 질문에 답하세요
                    반드시 json 형식으로만 답하세요

                    [컨텍스트]
                    {context_string}

                    [질문]
                    {user_query}

                    답변은 반드시 아래 형식의 json으로만 출력하세요
                    {
                        {
                            "recommended_product" : "상품명",
                            "reason":"추천사유"                            
                        }
                    }
                    ''')
print(raw_prompt)


당신은 이커머스 전문 AI 추천 어시스턴트입니다.
아래 컨텍스트를 참고하여 사용자의 질문에 답하세요
반드시 json 형식으로만 답하세요

[컨텍스트]
- 상품명:고성능 노트북 | 카테고리:가전- 상품명:사무용 랩탑 | 카테고리:가전- 상품명:미러리스 카메라 | 카테고리:가전

[질문]
코딩할 때 쓸만한 노트북 하나 추천해줘, 믿을만한 제품으로

답변은 반드시 아래 형식의 json으로만 출력하세요
{'recommended_product': '상품명', 'reason': '추천사유'}



## LLM 응답

In [5]:
import json
response = llm.generate(raw_prompt)
data = json.loads(response)
print(json.dumps(data, indent=2, ensure_ascii=False))

{
  "recommended_product": "사무용 랩탑",
  "reason": "코딩 작업에 필요한 기본 성능과 안정적인 사용성을 갖춘 사무용 랩탑을 추천합니다. 믿을만한 범용 제품이라 개발 환경(IDE, 브라우저, 문서 작업)을 무리 없이 사용하기 좋습니다."
}


# [2교시]

## 라우팅
- 들어온 질문을 보고 어느 경로로 보낼지 결정하는 단계

In [6]:
def route_question(question:str)->str:
    lower_question = question.lower()
    if any(keyword in lower_question for keyword in ['뉴스','기사','검색','찾아줘','최신','오늘']):
        return "news_tool"
    elif any(keyword in lower_question for keyword in ['계산','더하기','곱하기','합계','몇','얼마']):
        return "calculator_tool"
    elif any(keyword in lower_question for keyword in ['기억','기록','선호','메모','이전']):
        return "memory_tool"
    elif any(keyword in lower_question for keyword in ['추천','골라','비교']):
        return "llm_recomendation"
    else:
        return "general_llm"
sample_questions = [
    "3개 상품을 2개씩 주문하면 총 몇 개인가?"
    "나는 코딩용 노트북을 좋아한다는 점을 기억해줘."
    "AI 에이전트 뉴스 최신 기사 3개 찾아줘"
    "코딩할 때 쓸만한 노트북 추천해줘."
]
for question in sample_questions:
    print(f'질문 : {question} | 라우트 : {route_question(question)}')

질문 : 3개 상품을 2개씩 주문하면 총 몇 개인가?나는 코딩용 노트북을 좋아한다는 점을 기억해줘.AI 에이전트 뉴스 최신 기사 3개 찾아줘코딩할 때 쓸만한 노트북 추천해줘. | 라우트 : news_tool


## tool 활용

In [7]:
import json
import os
import re
from urllib.parse import quote
from urllib.request import Request,urlopen


def calculator_tool(text:str)->float:
    allowed_chars = set("0123456789+-*/().")
    if not set(text) <= allowed_chars: # text의 문자 중에 허용되지 않은 문자가 있다면
        raise valueError('허용되지 않은 문자가 포함되어 있습니다.')
    return eval(text)

def search_naver_news(query:str, display:int=3) -> list[dict]:
    pass

In [8]:
# 네이버 검색 API 예제 - 블로그 검색
import os
import re
import sys
import json
import html
import urllib.request
from datetime import datetime
from dotenv import load_dotenv
load_dotenv(override=True)

def _format_date(pubdate):
    return datetime.strptime(pubdate, "%a, %d %b %Y %H:%M:%S %z").strftime("%Y-%m-%d")

def _format_str(text):
    return html.unescape(re.sub(r'<[^>]+>',"",text))

client_id = os.getenv('NAVER_CLIENT_ID')
client_secret = os.getenv('NAVER_CLIENT_SECRET')

items = []
def search_naver_news(query:str, display:int=3)->list[dict]:
    encText = urllib.parse.quote(query)
    encText+= f'&display={display}&sort=date'
    url = "https://openapi.naver.com/v1/search/news?query=" + encText # JSON 결과    
    request = urllib.request.Request(url)
    request.add_header("X-Naver-Client-Id",client_id)
    request.add_header("X-Naver-Client-Secret",client_secret)


    response = urllib.request.urlopen(request)
    rescode = response.getcode()
    if(rescode==200):
        response_body = response.read().decode('utf-8')
        result = json.loads(response_body)

        for row in result.get('items'):
            items.append({
                'title':_format_str(row.get('title')),
                'content':_format_str(row.get('description')),                
                'date':_format_date(row.get('pubDate')),
                'link':row.get('link')
            })         
    return items

In [9]:
search_naver_news('AI 에이전트')

[{'title': '“사람처럼 드나드는 AI 에이전트... 인증 규격 등 ‘보안 프레임워크’...',
  'content': 'AI 에이전트의 안전한 도입과 운영을 위한 보안 프레임워크가 빠르게 마련돼야 한다는 지적이 나왔다. 사람을 대신해 내부 시스템에 들어와 업무를 수행하는 만큼, 접근 관리 등의 체계가 필요하다는 목소리다. 27일... ',
  'date': '2026-05-27',
  'link': 'http://www.boannews.com/media/view.asp?idx=143835&kind=2'},
 {'title': "경산시, 1421억 펀드로 키운 스타트업…글로벌 무대서 '기술력 입증'",
  'content': '더선한은 이번 선정을 계기로 독자적인 AI 에이전트 기술 고도화와 글로벌 시장 진출에 박차를 가할 예정이다. 대구대 기술창업 기반으로 설립된 이차전지 소재·부품·장비 기업 티씨엠에스는 경산시 펀드 연계... ',
  'date': '2026-05-27',
  'link': 'https://n.news.naver.com/mnews/article/030/0003431836?sid=102'},
 {'title': '[헬로AI] AI 확장의 마지막 퍼즐 ‘오케스트레이션’…실험을 넘어 운영...',
  'content': '많은 조직이 AI 에이전트를 사용하고 있다고 보고했지만, 작년에 실제로 상용화된 프로젝트는 10개 중 1개에 불과했다. 그 결과 좌절감이 커지고 있다. 수많은 시범 사업과 실험이 진행되지만, 그 영향력과 결과는... ',
  'date': '2026-05-27',
  'link': 'https://www.hellot.net/news/article.html?no=112813'}]

## 메모리 활용하기
- 이전대화나 사용자의 선호를 저장해서 다음 응답에 반영하는 기능

In [10]:
session_memory={}
def remember_preference(user_id:str, key:str, value:str)->None:
    if user_id not in session_memory:
        session_memory[user_id] = {}
    session_memory[user_id][key] = value

def get_preference(user_id:str, key:str, default:str | None = None) -> str | None:
    return session_memory.get(user_id,{}).get(key,default)

user_id = 'student-001'
remember_preference(user_id, 'category', '노트북')
remember_preference(user_id, 'usage', '코딩')

print(session_memory)

{'student-001': {'category': '노트북', 'usage': '코딩'}}


# [3교시]

### 프롬프트 템플릿  Agent tool rag에서 표준
- role:LLM이 어떤 역활을 할지 정함
- instruction:답변 규칙과 형식을 정리
- context:데이터베이스 검색결과처럼 답변에 참고할 실제 정보
- query:실제 질문


In [11]:
# 사용자 질문 -> LLM개입해서 사용자 의도를 파악해서
# 적절한 tool 호출 -> 라우터( Fast API )
# 수집한 정보를 -> LLM 전달
# 최종답변은 LLM

In [12]:
import os
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv(override=True)

api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    raise EnvironmentError('openai api key .....')

class OpenAILLM:
    def __init__(self,model:str = 'gpt-5.4-nano'):
        self.client = OpenAI(api_key=api_key)
        self.model = model
    def generate(self, prompt:str) -> str:
        response = self.client.chat.completions.create(
            model = self.model,
            messages =[
                {"role":"system", "content":"You are an ecomerce recommendation assistant, Return only valid JSON"},
                {"role":"user", "content":prompt}
            ],
            temperature=0,
            response_format={'type':'json_object'}
        )
        return response.choices[0].message.content
llm = OpenAILLM()
llm.model  

'gpt-5.4-nano'

In [13]:
# 데이터
do_search_results = [
    {"id":"p1", "name":"고성능 노트북","category":"가전"},
    {"id":"p2", "name":"사무용 랩탑","category":"가전"},
    {"id":"p3", "name":"미러리스 카메라","category":"가전"},
]
context_string = ""
for item in do_search_results:
    context_string += f"- 상품명:{item['name']} | 카테고리:{item['category']}"

In [14]:
from  textwrap import dedent  #  들여쓰기 indent를 제거
import json
# 프롬프트 템플릿 - 고정된 구조
# 프롬프트생성 + llm호출 + 파싱
def recommend_product(user_question:str, context:str) ->dict:    
    prompt = dedent(f'''
                        당신은  사용자의 질문에 정확히 응답하는 ai 시스템 입니다.
                        사용자의 질문과 context를 보고 질문의 의도에 맞게 출력하세요

                        [컨텍스트]
                        {context}

                        [질문]
                        {user_question}                        
                        
                        [출력]
                        답변은 반드시 아래와 같은 json형태로 
                        {
                            {
                                "assistant" : "출력내용",
                                "reason":"사유"                            
                            }
                        }

                        ''')
    response = llm.generate(prompt)
    data = json.loads(response)
    return json.dumps(data, indent=2, ensure_ascii=False)
    
    

### 라우팅
- 들어온 질문을 보고 어느 경로로 보낼지 결정하는 단계

In [15]:
def route_qeustion(question:str)->str:
    lower_question = question.lower()
    if any(keyword in lower_question for keyword in ['뉴스','기사','검색','찾아줘','최신','오늘']):
        return "news_tool"
    elif any(keyword in lower_question for keyword in ['계산','더하기','곱하기','합계','몇','얼마']):
        return "calculator_tool"
    elif any(keyword in lower_question for keyword in ['기억','기록','선호','메모','이전']):
        return "memory_tool"
    elif any(keyword in lower_question for keyword in ['추천','골라','비교']):
        return "llm_recommendation"
    else:
        return "general_llm"
sample_questions = [
    "3개 상품을 2개씩 주문하면 총 몇 개인가?",
    "나는 코딩용 노트북을 좋아한다는 점을 기억해줘.",
    "AI 에이전트 뉴스 최신 기사 3개 찾아줘.",
    "코딩할 때 쓸만한 노트북 추천해줘.",
]
for question in sample_questions:
    print(f'질문 : {question} | 라우트 : {route_qeustion(question)}')    

질문 : 3개 상품을 2개씩 주문하면 총 몇 개인가? | 라우트 : calculator_tool
질문 : 나는 코딩용 노트북을 좋아한다는 점을 기억해줘. | 라우트 : memory_tool
질문 : AI 에이전트 뉴스 최신 기사 3개 찾아줘. | 라우트 : news_tool
질문 : 코딩할 때 쓸만한 노트북 추천해줘. | 라우트 : llm_recommendation


## tool 활용

In [16]:
import json
import os
import re
from urllib.parse import quote
from urllib.request import Request,urlopen


def calcualtor_tool(text:str)->float:
    allowed_chars = set("0123456789+-*/(). ")
    if not set(text) <= allowed_chars: # text의 문자중에 허용되지 않은 문자가 있다면
        raise ValueError('허용되지 않은 문자가 포함되어 있습니다.')
    return eval(text)

In [17]:
# 네이버 검색 API 예제 - 블로그 검색
import os
import re
import sys
import json
import html
import urllib.request
from datetime import datetime
from dotenv import load_dotenv
load_dotenv(override=True)

def _format_date(pubdate):
    return datetime.strptime(pubdate, "%a, %d %b %Y %H:%M:%S %z").strftime("%Y-%m-%d")

def _format_str(text):
    return html.unescape(re.sub(r'<[^>]+>',"",text))

client_id = os.getenv('NAVER_CLIENT_ID')
client_secret = os.getenv('NAVER_CLIENT_SECRET')

items = []
def search_naver_news(query:str, display:int=3)->list[dict]:
    encText = urllib.parse.quote(query)
    encText+= f'&display={display}&sort=date'
    url = "https://openapi.naver.com/v1/search/news?query=" + encText # JSON 결과    
    request = urllib.request.Request(url)
    request.add_header("X-Naver-Client-Id",client_id)
    request.add_header("X-Naver-Client-Secret",client_secret)


    response = urllib.request.urlopen(request)
    rescode = response.getcode()
    if(rescode==200):
        response_body = response.read().decode('utf-8')
        result = json.loads(response_body)

        for row in result.get('items'):
            items.append({
                'title':_format_str(row.get('title')),
                'content':_format_str(row.get('description')),                
                'date':_format_date(row.get('pubDate')),
                'link':row.get('link')
            })         
    return items

In [18]:
print(calcualtor_tool('30*50'))
search_naver_news('AI 에이전트')

1500


[{'title': '“사람처럼 드나드는 AI 에이전트... 인증 규격 등 ‘보안 프레임워크’...',
  'content': 'AI 에이전트의 안전한 도입과 운영을 위한 보안 프레임워크가 빠르게 마련돼야 한다는 지적이 나왔다. 사람을 대신해 내부 시스템에 들어와 업무를 수행하는 만큼, 접근 관리 등의 체계가 필요하다는 목소리다. 27일... ',
  'date': '2026-05-27',
  'link': 'http://www.boannews.com/media/view.asp?idx=143835&kind=2'},
 {'title': "경산시, 1421억 펀드로 키운 스타트업…글로벌 무대서 '기술력 입증'",
  'content': '더선한은 이번 선정을 계기로 독자적인 AI 에이전트 기술 고도화와 글로벌 시장 진출에 박차를 가할 예정이다. 대구대 기술창업 기반으로 설립된 이차전지 소재·부품·장비 기업 티씨엠에스는 경산시 펀드 연계... ',
  'date': '2026-05-27',
  'link': 'https://n.news.naver.com/mnews/article/030/0003431836?sid=102'},
 {'title': '[헬로AI] AI 확장의 마지막 퍼즐 ‘오케스트레이션’…실험을 넘어 운영...',
  'content': '많은 조직이 AI 에이전트를 사용하고 있다고 보고했지만, 작년에 실제로 상용화된 프로젝트는 10개 중 1개에 불과했다. 그 결과 좌절감이 커지고 있다. 수많은 시범 사업과 실험이 진행되지만, 그 영향력과 결과는... ',
  'date': '2026-05-27',
  'link': 'https://www.hellot.net/news/article.html?no=112813'}]

### 메모리 활용하기
- 이전대화나 사용자의 선호를 저장해서 다음 응답에 반영하는 기능

In [19]:
session_memory={}
def remember_preference(user_id:str, key:str, value:str)->None:
    if user_id not in session_memory:
        session_memory[user_id] = {}
    session_memory[user_id][key] = value

def get_preference(user_id:str, key:str, default:str | None = None) -> str | None:
    return session_memory.get(user_id,{}).get(key,default)

user_id = 'student-001'
remember_preference(user_id, 'category','노트북')
remember_preference(user_id, 'usage','코딩')

print(session_memory)

{'student-001': {'category': '노트북', 'usage': '코딩'}}


### 라우터 + 도구 + 메모리 통합
- 라우팅 규칙, 계산도구, 네이버뉴스도구, 메모리 저장을 하나의 흐름으로 연결--> Agent의 기본 형태
- 먼저 질문을 분류하고 그 분류 결과에 맞는 도구를 호출한뒤 필요하면 메모리까지 갱신
- 최종 결과를 llm에 전달해서 답변을 생성

In [20]:
def extract_math_expression(question: str) -> str:
    match = re.search(r"[0-9\s\+\-\*\/\(\)\.]+", question)
    if not match:
        raise ValueError("계산식을 찾을 수 없습니다.")
    expression = match.group(0).strip()
    if not expression:
        raise ValueError("계산식이 비어 있습니다.")
    return expression

def extract_news_query(question: str) -> str:
    cleaned = re.sub(r"뉴스|기사|검색|알려줘|찾아줘|추천해줘|좀|최근|최신|오늘", " ", question)
    cleaned = re.sub(r"\d+\s*개?", " ", cleaned)
    cleaned = re.sub(r"[^0-9A-Za-z가-힣\s]", " ", cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned or question

def route_and_execute(question: str) -> dict:
    route = route_qeustion(question)

    if route == "news_tool":
        news_query = extract_news_query(question)
        news_result = search_naver_news(news_query, display=3)
        news_result = recommend_product('뉴스 요약해줘',news_result)
        return {
            "route": route,
            "tool": "search_naver_news",
            "input": news_query,
            "result": news_result,
        }

    if route == "calculator_tool":
        expression = extract_math_expression(question)
        result = calcualtor_tool(expression)
        return {
            "route": route,
            "tool": "calculator_tool",
            "input": expression,
            "result": result,
        }

    if route == "memory_tool":
        remember_preference("student-001", "last_question", question)
        return {
            "route": route,
            "tool": "memory_tool",
            "input": question,
            "result": get_preference("student-001", "last_question"),
        }

    if route == "llm_recommendation":
        recommendation = recommend_product(question, context_string)
        return {
            "route": route,
            "tool": "llm_recommendation",
            "input": question,
            "result": recommendation,
        }

    return {
        "route": route,
        "tool": "general_llm",
        "input": question,
        "result": question,
    }

In [21]:
demo_questions = [
    # '3 + 2 * 4는 얼마야',
    '어제 오늘 사고뉴스 5개 찾아줘',
    '점심메뉴 추천해줘',
    '연구용 노트북 추천해줘',
]
print(json.loads(route_and_execute('어제 오늘 사고뉴스 5개 찾아줘')['result'])['assistant'])
print(json.loads(route_and_execute('어제 오늘 사고뉴스 5개 찾아줘')['result'])['reason'])
# for question in demo_questions:
#     result = route_and_execute(question)        
#     print(f'\n질문:{question}')
#     print(f'라우트:{result["route"]}')
#     print(f'도구:{result["tool"]}')
#     print(f'결과:{result["result"]}')
#     print(f'결과:{result["result"]["reason"]}')

제공된 뉴스들은 전반적으로 (1) AI 에이전트의 안전한 도입·운영을 위한 보안 프레임워크 필요성, (2) AI 에이전트 기술 고도화 및 글로벌 진출을 추진하는 스타트업/지역 투자 성과, (3) AI 확장의 다음 단계로 ‘오케스트레이션(운영·조율)’이 중요하다는 내용, (4) 서소문 고가차도 붕괴 사고 관련 조문 및 철거·안전관리 절차 점검, (5) 사고 시기와 관련한 정치권 논란 등으로 요약됩니다.
context에 포함된 기사들의 주제를 기준으로 공통 흐름(기술/산업, AI 운영, 사고 수사, 정치 공방)으로 묶어 간단히 요약했기 때문입니다.


In [22]:
# 1. 뉴스검색의 경우... 출력을 조정
# 2. 추천의 경우, 프롬프트가 현재 이커머스로 되어있는데 -> 일반적인 프롬프트로 변경

# [4교시]

In [63]:
import os
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv(override=True)

api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    raise EnvironmentError('openai api key .....')
class OpenAILLM:
    def __init__(self,model:str = 'gpt-5.4-nano'):
        self.client = OpenAI(api_key=api_key)
        self.model = model
    def generate(self, prompt:str) -> str:
        response = self.client.chat.completions.create(
            model = self.model,
            messages =[
                {"role":"system", "content":"너는 사용자의 질문에 친철하고 정확하게 답변하는 시스템 입니다., Return only valid JSON"},
                {"role":"user", "content":prompt}
            ],
            temperature=0,            
        )
        return response.choices[0].message.content
llm = OpenAILLM()    

In [64]:
llm.generate('어제 삼성전자 주식 종가 알려줘')

'{\n  "error": "요청하신 \'어제 삼성전자 주식 종가\'를 실시간으로 조회할 수 없습니다.",\n  "details": {\n    "need_from_user": [\n      "어느 거래소 기준인가요? (예: 한국거래소/KRX)",\n      "어제 날짜를 확인해 주세요. (예: 2026-05-26 기준인지)"\n    ],\n    "suggestion": "원하시면 \'기준 날짜(YYYY-MM-DD)\'와 \'거래소(KRX)\'를 알려주시면, 그 날짜의 종가를 확인하는 방법(또는 확인 가능한 데이터 소스/조회 절차)을 안내해 드릴게요."\n  }\n}'

In [73]:
import json
from  textwrap import dedent  #  들여쓰기 indent를 제거
query = '어제 삼성전자 주식종가 를 조회하고 해당 종가와 비슷한종목을 뉴스에서 검색해서 요약하고 오늘날씨에 맞는 외출복을 추천하고 데이트 장소 추천해'
def routerLLM(query):
    prompt = dedent(f"""
    사용자의 질문 의도를 분석하세요.

    질문에 포함된 의도가 여러 개이면
    반드시 모든 의도를 각각 분리하여 출력하세요.

    예를 들어:
    - 주식 + 뉴스 + 날씨
    - 뉴스 + 검색
    - 계산 + 주식

    처럼 복합 질문이면
    반드시 여러 개의 JSON 객체를 배열로 출력해야 합니다.

    question_type 별로 tool_query를 생성하세요.
    tool_query는 반드시 키워드중심으로 생성하세요 llm에 전달하는 문구가 아님을 명심해서 추론을하지말고 검색용 키워드로 매칭해주세요    
    뉴스는 api검색할수 있는 키워드중심으로,
    주식은 종목명만 추출하세요             
    날씨도 검색용 키워드로 추출하세요                               

    지원 가능한 question_type 예시:
    - 날씨
    - 뉴스
    - 주식
    - 검색
    - 계산
    - 지도

    [중요 규칙]
    - 질문에 포함된 모든 의도를 누락 없이 출력
    - 반드시 JSON 배열(Array) 형식으로 출력
    - 하나만 출력 금지
    - 설명문 금지
    - markdown 금지
    - ```json 금지

    [예시 입력]
    어제 삼성전자 종가 알려주고 관련 뉴스와 오늘 날씨도 알려줘

    [예시 출력]
    [
        {{
            "question_type": "주식",
            "tool_query": "삼성전자",
            "reason": "삼성전자 종가 요청"
        }},
        {{
            "question_type": "뉴스",
            "tool_query": "삼성전자 관련 최근 뉴스 검색",
            "reason": "관련 뉴스 요청"
        }},
        {{
            "question_type": "날씨",
            "tool_query": "오늘 날씨 조회",
            "reason": "날씨 요청"
        }}
    ]

    [질문]
    {query}

    [출력]
    반드시 유효한 JSON 배열만 출력하세요.
    """)

    result = llm.generate(prompt)
    json_result = json.loads(result)
    return json_result

In [74]:
# 네이버 검색 API 예제 - 블로그 검색
import os
import re
import sys
import json
import html
import urllib.request
from datetime import datetime
from dotenv import load_dotenv
load_dotenv(override=True)

def _format_date(pubdate):
    return datetime.strptime(pubdate, "%a, %d %b %Y %H:%M:%S %z").strftime("%Y-%m-%d")

def _format_str(text):
    return html.unescape(re.sub(r'<[^>]+>',"",text))

client_id = os.getenv('NAVER_CLIENT_ID')
client_secret = os.getenv('NAVER_CLIENT_SECRET')

items = []
def search_naver_news(query:str, display:int=3)->list[dict]:
    encText = urllib.parse.quote(query)
    encText+= f'&display={display}&sort=date'
    url = "https://openapi.naver.com/v1/search/news?query=" + encText # JSON 결과    
    request = urllib.request.Request(url)
    request.add_header("X-Naver-Client-Id",client_id)
    request.add_header("X-Naver-Client-Secret",client_secret)


    response = urllib.request.urlopen(request)
    rescode = response.getcode()    
    if(rescode==200):
        response_body = response.read().decode('utf-8')
        result = json.loads(response_body)        
        for row in result.get('items'):
            items.append({
                'title':_format_str(row.get('title')),
                'content':_format_str(row.get('description')),                
                'date':_format_date(row.get('pubDate')),
                'link':row.get('link')
            })         
    return items

In [75]:
search_naver_news('삼성전자 관련 최근 뉴스 검색')

[{'title': '‘N% 성과급’ 요구 산업계 전반 번져…재계 “투자 여력 위축” 우려',
  'content': '독자 유형별 맞춤 뉴스 6개를 선별해 제공합니다. ■ 성과급 전쟁의 산업 전반 확산: 삼성전자(005930) 노사... 정보 검색을 넘어 전문 실무를 직접 수행하는 단계로 빠르게 접어들었다. 현대차 에이미는 경쟁사 제품... ',
  'date': '2026-05-22',
  'link': 'https://n.news.naver.com/mnews/article/011/0004623517?sid=101'},
 {'title': '[Who Is ?] 표경원 애경케미칼 대표이사 부사장',
  'content': '애경케미칼은 최근 3년간 연구개발비를 꾸준히 늘렸다. 2023년 210억7400만 원, 2024년 210억9900만 원, 2025년... <연합뉴스> 1995년 삼성자동차에 입사했다. 1999년 삼성전자로 이동했다. 2002년 베인앤컴퍼니(Bain & Company)... ',
  'date': '2026-05-22',
  'link': 'https://www.businesspost.co.kr/BP?command=article_view&num=437623'},
 {'title': '[K-VIBE] 전태수의 웹 3.0 이야기…참여로 여는 디지털 기본 소득의 시대',
  'content': '연합뉴스 동포·다문화부 K컬처팀은 독자 여러분께 새로운 시선으로 한국 문화를 바라보는 데 도움이 되고자 전문가 칼럼 시리즈를 준비했습니다. 시리즈는 매주 게재합니다. 삼성전자 파업 이슈는 일단락됐지만, 그... ',
  'date': '2026-05-21',
  'link': 'https://n.news.naver.com/mnews/article/001/0016092628?sid=102'}]

# [5교시]

In [76]:
for row in json_result:
    if row.get('question_type') == '뉴스':
        print('검색어', row.get('tool_query'))
        print(search_naver_news(row.get('tool_query')))

NameError: name 'json_result' is not defined

In [77]:
# tool excution
tool_results = []
def excute_tools(router_result):
    for row in router_result:
        query_type = row.get('question_type')
        print(f'tool : {query_type}')
        if query_type == '뉴스':        
            query = row.get('tool_query')
            news_result = search_naver_news(row.get('tool_query'))
            tool_results.append({
                'question_type' : '뉴스',
                'query' : query,
                'result' : news_result
            })
        # other tools
    return tool_results            

In [78]:
def make_message(user_query:str, tool_results:list[dict]):
    prompt = f'''
    너는 다양한 도구의 결과를 종합해서 사용자 질문에 답변하는 ai assistant 입니다.

    [사용자질문]
    {user_query}

    [도구실행결과]
    {json.dumps(tool_results, ensure_ascii=False, indent=2)}

    [지침]
    - tool 결과를 기반으로 답변하세요
    - 필요한 경우 여러 tool 결과를 종합하세요
    - 지도 주식 날씨 검색 추천등 다양한 데이터가 포함될수 있습니다.
    - tool 결과내에 있는 내용에 우선순위를 높게해서 해당 데이터기반으로 답변하세요
    '''
    return [
                {"role":"system", "content":"너는 여러 tool결과를 조합해서 최종 답변을생성하는 ai agent 입니다."},
                {"role":"user", "content":prompt}
            ]
    
    
# final llm
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
def final_llm_openai(user_query:str, tool_results:list[dict]):    
    result = client.chat.completions.create(
            model = 'gpt-5.4-nano',
            messages =make_message(user_query,tool_results),
            temperature=0.3            
        )
    return result.choices[0].message.content

### 파이프라인
- 사용자 질문
- 질문의 의도를 분석하는 llm 호출
- 해당 tools을 호출해서 결과 수집
- 최종 llm에게 전달
- 최종 답변 생성

In [79]:
query = '붕괴사고에 대해서'
router_result = routerLLM(query)
tool_results = excute_tools(router_result)
final_result = final_llm_openai(query,tool_results)
print(final_result)

tool : 뉴스
붕괴사고에 대해 말씀하신 것으로 보이는데, 제공된 도구 결과에서는 **서울 서소문 고가차도(철거 현장) 붕괴사고** 관련 내용이 가장 직접적으로 확인됩니다. (다른 “붕괴사고” 뉴스들은 검색 결과에 포함되지 않았습니다.)

## 1) 서울 서소문 고가차도 붕괴사고(철거 현장) 개요
- **사고 내용**: 서울 **서소문 고가차도 철거 현장**에서 **붕괴 사고**가 발생했습니다.
- **대응/지원**: 서울시는 **유가족과 부상자**를 위한 지원을 “전방위적으로” 시행하는 것으로 보도되었습니다.
- **복구 일정**: 국토교통부는 **주중 복구 목표**를 두고 복구에 총력을 기울인다고 전해졌습니다.  
  - 출처: 이투데이(2026-05-27) https://www.etoday.co.kr/news/view/2588316

## 2) 사고 원인/안전관리 “허점” 지적(후속 논의)
- 사고를 계기로 **시설물 안전관리 체계와 비상 대응 매뉴얼을 재점검**해야 한다는 지적이 나왔습니다.
- 특히 기사에서는 **철로를 횡단하거나 하부를 통과하는 교량 시설물**에 대해 **선제 통제(사전 통제) 강화** 필요성을 언급합니다.  
  - 출처: 경북일보(2026-05-27) https://www.kyongbuk.co.kr/news/articleView.html?idxno=4073909

---

원하시면, 질문을 조금만 더 구체화해 주세요. 예를 들어 **“어느 붕괴사고(서소문 말고 다른 지역/사건)?”**, 또는 **“원인·피해 규모·수사/조사 진행 상황”** 중 무엇을 중심으로 정리해드릴까요?


# [6교시]

In [80]:
# 1. planner / router
# 2. tool excuter
# 3. memory / retrieval
# 4. LLM
# 5. final response

### 모델을 허깅페이스계열의 모델로 변경

In [81]:
from huggingface_hub import notebook_login

os.environ.pop('HF_TOKEN', None)
notebook_login()

In [82]:
from huggingface_hub import get_token, whoami

HF_TOKEN = get_token()
if not HF_TOKEN:
    raise RuntimeError('Hugging Face 토큰이 저장되지 않았습니다. 로그인 셀을 다시 실행하세요.')

info = whoami(token=HF_TOKEN)
print('logged in user:', info.get('name'))
print('token prefix:', HF_TOKEN[:6] + '***')

logged in user: jiyong0923
token prefix: hf_aMw***


### 모델 변경

In [83]:
from transformers import AutoModelForCausalLM, AutoTokenizer
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# [7교시]

In [84]:
def final_llm_qween(user_query:str, tool_results:list[dict]):    
    text = tokenizer.apply_chat_template(
        make_message(user_query,tool_results),
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response

In [85]:
query = '붕괴사고에 대해서'
router_result = routerLLM(query)
tool_results = excute_tools(router_result)  
final_result = final_llm_qween(query,tool_results)
print(final_result)

tool : 뉴스
네, 네트워크를 통해 여러 도구의 결과를 종합하여 답변하겠습니다. 

1. **뉴스 결과**:
   - **서울시**: "서울 서소문 고가차도 철거 현장 붕괴 사고를 계기로 건설공사 발주자 책임을 강화하는 '건설안전특별법' 제정 논의가 다시 주목받고 있다."
   
   - **국토부**: "서울 서소문 고가차도 철거 현장 붕괴 사고를 계기로 건설공사 발주자 책임을 강화하는 '건설안전특별법' 제정 논의가 다시 주목받고 있다."

   - **삼성전자**: "삼성전자(005930) 노사 업계 전반에 대한 테스트 결과를 보여줍니다."
   
   - **현대차 에이미**: "현대차 에이미(000030) 경쟁사 제품의 안전 관리 방안에 대한 정보입니다."

2. **기술 결과**:
   - **네트워크**: "네트워크에 의해 수집된 정보를 기반으로 다양한 도구의 결과를 분석하고 있습니다."
   
   - **데이터 검색**: "데이터 검색 기능을 활용하여 다양한 도구의 결과를 분석합니다."
   
   - **AI 도구**: "AI 도구를 사용하여 다양한 도구의 결과를 종합적으로 분석하고 있습니다."
   
   - **물리적 장치**: "물리적 장치를 이용하여 다양한 도구의 결과를 분석하고 있습니다."
   
   - **기타**: "기타 기술이나 서비스를 활용한 분석 결과를 제공합니다."

3. **물리적 장치 결과**:
   - **GPS 위치**: "GPS 위치 기반으로 많은 도구의 결과를 분석합니다."
   
   - **위도/경도**: "위도 및 경도를 기반으로 많은 도구의 결과를 분석합니다."
   
   - **모든 도구**: "모든 도구를 포함하여 여러 가지 도구의 결과를 분석합니다."

이러한 다양한 데이터를 종합하여 여러 가지 가능한 답변을 제공할 수 있습니다.


In [87]:
query = '오늘 최신 뉴스에서 가장 중요한 내용을 요약해서 100자 이내로 출력'
router_result = routerLLM(query)
tool_results = excute_tools(router_result)  
final_result = final_llm_qween(query,tool_results)
print(final_result)

tool : 뉴스
네, 저는 여러 도구의 결과를 종합하여 최종 답변을 생성하도록 합니다. 

아래는 최종 답변 예시입니다:

---

"네, 오늘 최신 뉴스에서 가장 중요한 내용을 요약해서 100자 이내로 발표하겠습니다."

---

**1. **"‘N% 성과급’ 요구 산업계 전반 번져…"**
   - **Result**: 삼성전자, 애플, LG 등이 2026년까지 성과급을 유지하고 있는 것을 확인하였습니다. 또한, 신규 사업 영입, 투자 등을 통해 경쟁력을 확보하고 있는 것으로 보입니다. 
   
   **2. **"‘N% 성과급’ 요구 산업계 전반 번져…"**
   - **Result**: 2026년까지 삼성전자와 LG의 성과급이 유지되고 있다는 사실을 확인하였습니다. 또한, 애플, 현대차, 현대모비스 등의 업체들이 신규 사업 영입 및 투자에 대한 계획을 세웠습니다.
   
   **3. **"‘N% 성과급’ 요구 산업계 전반 번져…"**
   - **Result**: 2026년까지 삼성전자와 LG의 성과급이 유지되고 있으며, 삼성전자는 신규 사업 영입 및 투자에 대한 계획을 세워왔습니다.
   
   **4. **"‘N% 성과급’ 요구 산업계 전반 번져…"**
   - **Result**: 삼성전자와 LG의 성과급은 2026년까지 유지되고 있으며, 삼성전자는 신규 사업 영입 및 투자에 대한 계획을 세워왔습니다.
   
   **5. **"‘N% 성과급’ 요구 산업계 전반 번져…"**
   - **Result**: 삼성전자와 LG의 성과급은 2026년까지 유지되고 있으며, 삼성전자는 신규 사업 영입 및 투자에 대한 계획을 세워왔습니다.

---

**지침:** 각 도구의 결과를 종합하여 최종 답변을 생성하려면, 아래의 순서로 진행하면 됩니다:
1. **최신 �


In [88]:
query = '오늘 최신 뉴스에서 가장 중요한 내용을 요약해서 100자 이내로 출력'
router_result = routerLLM(query)
tool_results = excute_tools(router_result)  
final_result = final_llm_openai(query,tool_results)
print(final_result)

tool : 뉴스
서울 서소문 고가차도 붕괴: 3명 사망·3명 부상, 서울시 지원·국토부 주중 복구 목표, 경찰·검찰 전담 수사 착수.


# [8교시]

In [3]:
!pip install xmltodict

In [2]:
# 날씨 조회 api
import requests
from datetime import datetime
import xmltodict
import os
from dotenv import load_dotenv
load_dotenv(override=True)

weather_api = os.getenv('WEATHER_API')
print(f'weather_api : {weather_api[:10]}...')

def get_current_date():
    current_date = datetime.now().date()
    return current_date.strftime("%Y%m%d")

def get_current_hour():
    now = datetime.now()
    return datetime.now().strftime("%H%M")

int_to_weather = {
    "0": "맑음",
    "1": "비",
    "2": "비/눈",
    "3": "눈",
    "5": "빗방울",
    "6": "빗방울눈날림",
    "7": "눈날림"
}

def forecast(params):
    url = 'http://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getVilageFcst' # 초단기예보
    # 값 요청 (웹 브라우저 서버에서 요청 - url주소와 파라미터)
    res = requests.get(url, params)
    print(f'res : {res}')
    #XML -> 딕셔너리
    xml_data = res.text
    dict_data = xmltodict.parse(xml_data)

    for item in dict_data['response']['body']['items']['item']:
        if item['category'] == 'T1H':
            temp = item['obsrValue']
        # 강수형태: 없음(0), 비(1), 비/눈(2), 눈(3), 빗방울(5), 빗방울눈날림(6), 눈날림(7)
        if item['category'] == 'PTY':
            sky = item['obsrValue']
            
    sky = int_to_weather[sky]
    
    return temp, sky

keys = weather_api

params ={'serviceKey' : keys, 
         'pageNo' : '1', 
         'numOfRows' : '10', 
         'dataType' : 'XML', 
         'base_date' : get_current_date(), 
         'base_time' : get_current_hour(), 
         'nx' : '55', 
         'ny' : '127' }

forecast(params)

weather_api : d83644e531...
res : <Response [401]>


ExpatError: syntax error: line 1, column 0

In [12]:
import os
import requests
import xmltodict
from dotenv import load_dotenv

# .env 파일 로드
load_dotenv(override=True)

# 1. 서울시 API 키 가져오기 (.env 파일에 SEOUL_API 변수 세팅 필요)
seoul_api_key = os.getenv('SEOUL_API')

def get_seoul_city_data(params):
    # 2. URL 조립: 서울시 API는 params를 쿼리스트링이 아닌 URL 경로(Path)에 직접 넣습니다.
    url = f"http://openapi.seoul.go.kr:8088/{params['KEY']}/{params['TYPE']}/{params['SERVICE']}/{params['START_INDEX']}/{params['END_INDEX']}/{params['AREA_NM']}"
    
    # URL 완성 후 GET 요청
    res = requests.get(url)
    print(f'응답 상태 코드 : {res.status_code}')
    
    if res.status_code != 200:
        print("API 요청에 실패했습니다. 키 값이나 네트워크를 확인하세요.")
        return None, None, None

    xml_data = res.text
    
    try:
        dict_data = xmltodict.parse(xml_data)
        
        # 3. 정상 처리 확인 (결과 코드가 INFO-000 이면 정상)
        result_code = dict_data['SeoulRtd.citydata']['RESULT']['RESULT.CODE']
        if result_code != 'INFO-000':
            error_msg = dict_data['SeoulRtd.citydata']['RESULT']['RESULT.MESSAGE']
            print(f"[-] API 호출 오류: {error_msg}")
            return None, None, None

        # 4. 데이터 추출
        city_data = dict_data['SeoulRtd.citydata']['CITYDATA']
        
        # 인구 혼잡도 데이터
        congest_lvl = city_data['LIVE_PPLTN_STTS']['LIVE_PPLTN_STTS']['AREA_CONGEST_LVL']
        
        # 날씨 데이터 (기온, 강수 형태)
        weather = city_data['WEATHER_STTS']['WEATHER_STTS']
        temp = weather['TEMP']
        sky = weather['PRECPT_TYPE']
        
        return congest_lvl, temp, sky
        
    except Exception as e:
        print(f"데이터 파싱 중 에러 발생: {e}")
        return None, None, None


# 기존 코드처럼 파라미터 딕셔너리 셋업
params = {
    'KEY': seoul_api_key, 
    'TYPE': 'xml', 
    'SERVICE': 'citydata', 
    'START_INDEX': '1', 
    'END_INDEX': '5', 
    'AREA_NM': '광화문·덕수궁'  # 다른 지역(예: 강남 MICE 관광특구 등)으로 변경 가능
}

print(f"요청 장소: {params['AREA_NM']} 데이터 조회 중...\n")

# 함수 호출
congest_lvl, temp, sky = get_seoul_city_data(params)

# 결과 출력
if congest_lvl and temp and sky:
    print("-" * 40)
    print(f"📍 장소: {params['AREA_NM']}")
    print(f"🚶‍♂️ 현재 혼잡도: {congest_lvl}")
    print(f"🌡️ 현재 기온: {temp}°C")
    print(f"☁️ 현재 날씨: {sky}")
    print("-" * 40)

요청 장소: 광화문·덕수궁 데이터 조회 중...

응답 상태 코드 : 200
----------------------------------------
📍 장소: 광화문·덕수궁
🚶‍♂️ 현재 혼잡도: 약간 붐빔
🌡️ 현재 기온: 24.0°C
☁️ 현재 날씨: 없음
----------------------------------------


In [14]:
import os
import requests
import xmltodict

def seoul_city_data_tool(area_nm: str) -> dict:
    """서울시 특정 장소의 혼잡도와 날씨를 조회하는 도구"""
    seoul_api_key = os.getenv('SEOUL_API')
    
    # LLM이 이상한 공백을 넣을 수도 있으니 정리
    area_nm = area_nm.strip() 
    
    url = f"http://openapi.seoul.go.kr:8088/{seoul_api_key}/xml/citydata/1/5/{area_nm}"
    res = requests.get(url)
    
    if res.status_code != 200:
        return {"error": "API 요청에 실패했습니다."}

    try:
        dict_data = xmltodict.parse(res.text)
        result_code = dict_data['SeoulRtd.citydata']['RESULT']['RESULT.CODE']
        
        if result_code != 'INFO-000':
            return {"error": dict_data['SeoulRtd.citydata']['RESULT']['RESULT.MESSAGE']}

        city_data = dict_data['SeoulRtd.citydata']['CITYDATA']
        congest_lvl = city_data['LIVE_PPLTN_STTS']['LIVE_PPLTN_STTS']['AREA_CONGEST_LVL']
        weather = city_data['WEATHER_STTS']['WEATHER_STTS']
        temp = weather['TEMP']
        sky = weather['PRECPT_TYPE']
        
        return {
            "장소": area_nm,
            "인구혼잡도": congest_lvl,
            "현재기온": f"{temp}도",
            "강수형태": sky
        }
    except Exception as e:
        return {"error": f"데이터를 찾을 수 없거나 파싱 오류 발생: {e}"}

In [15]:
import json
from textwrap import dedent

def routerLLM_v2(query):
    prompt = dedent(f"""
    사용자의 질문 의도를 분석하세요.
    질문에 포함된 의도가 여러 개이면 반드시 모든 의도를 각각 분리하여 JSON 배열로 출력하세요.

    question_type 별로 tool_query를 생성하세요.
    tool_query는 반드시 키워드 중심으로 생성하세요.

    지원 가능한 question_type 예시 및 규칙:
    - 뉴스 : api 검색할 수 있는 키워드 중심으로 추출
    - 주식 : 종목명만 추출
    - 서울데이터 : 사용자가 서울의 특정 장소(예: 광화문, 강남역 등)의 혼잡도, 사람 많은지, 날씨 등을 물어볼 때 사용. tool_query에는 '광화문·덕수궁', '강남역' 등 "정확한 장소명"만 추출하세요.

    [예시 입력]
    광화문·덕수궁 사람 많아? 그리고 관련 뉴스도 찾아줘

    [예시 출력]
    [
        {{
            "question_type": "서울데이터",
            "tool_query": "광화문·덕수궁",
            "reason": "특정 장소의 혼잡도 요청"
        }},
        {{
            "question_type": "뉴스",
            "tool_query": "광화문 덕수궁",
            "reason": "관련 뉴스 검색 요청"
        }}
    ]

    [질문]
    {query}

    반드시 유효한 JSON 배열만 출력하세요. (markdown 금지)
    """)

    result = llm.generate(prompt)
    return json.loads(result)

In [16]:
def excute_tools_v2(router_result):
    tool_results = []
    for row in router_result:
        query_type = row.get('question_type')
        query = row.get('tool_query')
        
        print(f'🛠️ 도구 실행 중 : [{query_type}] -> 검색어: {query}')
        
        if query_type == '뉴스':        
            news_result = search_naver_news(query)
            tool_results.append({
                'question_type': '뉴스',
                'query': query,
                'result': news_result
            })
        elif query_type == '서울데이터':
            # 여기서 방금 만든 서울시 API 도구를 호출합니다!
            seoul_result = seoul_city_data_tool(query)
            tool_results.append({
                'question_type': '서울데이터',
                'query': query,
                'result': seoul_result
            })
            
    return tool_results

In [18]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# 1. 환경 변수 로드 (.env 파일에서 OPENAI_API_KEY 가져오기)
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("⚠️ OPENAI_API_KEY가 세팅되지 않았습니다. .env 파일을 다시 확인해 주세요.")
else:
    # 2. OpenAILLM 클래스 정의 (수업 시간에 사용하신 모델명 기준)
    class OpenAILLM:
        def __init__(self, model: str = 'gpt-5.4-nano'): # 필요시 'gpt-4o-mini' 등으로 변경 가능
            self.client = OpenAI(api_key=api_key)
            self.model = model
            
        def generate(self, prompt: str) -> str:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": "너는 사용자의 질문에 친절하고 정확하게 답변하는 시스템입니다."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0,
            )
            return response.choices[0].message.content

    # 3. llm 객체 생성 (에러의 원인이었던 변수 부활!)
    llm = OpenAILLM()

    # 4. 정상 작동 확인용 테스트
    try:
        test_response = llm.generate("안녕? 연결 테스트 중이야. 짧게 인사해 줘.")
        print("✅ llm 객체 생성 및 연결 완료! API 테스트 응답:", test_response)
    except Exception as e:
        print("❌ API 호출 중 에러가 발생했습니다:", e)

✅ llm 객체 생성 및 연결 완료! API 테스트 응답: 안녕하세요! 연결 테스트 중이시군요—잘 들리고 있어요. 😊


In [ ]:
import json
import os
from openai import OpenAI

# 1. 프롬프트 메시지 생성 함수 복구 (수업 코드와 동일)
def make_message(user_query:str, tool_results:list[dict]):
    prompt = f'''
    너는 다양한 도구의 결과를 종합해서 사용자 질문에 답변하는 ai assistant 입니다.

    [사용자질문]
    {user_query}

    [도구실행결과]
    {json.dumps(tool_results, ensure_ascii=False, indent=2)}

    [지침]
    - tool 결과를 기반으로 답변하세요
    - 필요한 경우 여러 tool 결과를 종합하세요
    - 지도 주식 날씨 검색 추천등 다양한 데이터가 포함될수 있습니다.
    - tool 결과내에 있는 내용에 우선순위를 높게해서 해당 데이터기반으로 답변하세요
    '''
    return [
        {"role":"system", "content":"너는 여러 tool결과를 조합해서 최종 답변을생성하는 ai agent 입니다."},
        {"role":"user", "content":prompt}
    ]
        
# 2. 최종 답변 생성 함수 복구
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

def final_llm_openai(user_query:str, tool_results:list[dict]):    
    result = client.chat.completions.create(
            model = 'gpt-5.4-nano',
            messages = make_message(user_query, tool_results),
            temperature=0.3            
        )
    return result.choices[0].message.content

# 3. 모은 자료를 최종 출력
print("3. 최종 결과 집계 중...\n")
print("=" * 50)
final_result = final_llm_openai(user_query, tool_results)
print(final_result)
print("=" * 50)

3. 최종 LLM이 결과 종합 중...

현재 **광화문·덕수궁 일대는 사람 많음이 아니라 ‘보통’**으로 나와요.  
또한 **날씨는 현재 24.0도, 강수 없음(비/눈 없음)** 입니다.

## 오늘 광화문 관련 뉴스 3개 요약
1) **[내일날씨] 출근길 곳곳 비…오후부터 맑아져 초여름 더위**
- 부처님 오신날 대체공휴일(25일) 이후로 여름 더위가 이어지는 가운데, **광화문광장 일대에서 비가 내리거나(새벽부터) 이후 맑아질 가능성**과 기온 상승(최고 30도 언급)이 전해졌어요.

2) **정부 “나무호 타격 비행체, 이란 대함미사일 가능성 커…”**
- 외교부 브리핑에서 **나무호 피격 사건 조사 결과**를 공개하며, **비행체가 이란의 대함미사일일 가능성** 등을 언급한 내용입니다.

3) **[퇴근길 날씨] 수도권 중심으로 비…강원 동해안 짙은 안개**
- 기상청 예보 기준으로 **수도권(서울·인천·경기) 중심 비 가능성**이 있고, **강원 동해안은 짙은 안개**가 예상된다는 내용이에요.

원하시면 “지금 광화문광장/덕수궁 주변”을 더 세분해서(예: 광화문광장 vs 덕수궁 돌담길) 혼잡도/체감 날씨도 다시 정리해드릴게요.
